In [ ]:
import os
import sys

ROOT_PATH = '../../'
os.chdir(ROOT_PATH)

sys.path.append(os.path.abspath(ROOT_PATH))

lib_path = os.path.abspath(os.path.join(ROOT_PATH, 'lib'))
if lib_path not in sys.path:
  sys.path.append(lib_path)

In [2]:
from pathlib import Path

import torch
import numpy as np
import cv2
from tqdm.notebook import tqdm

from lib.depth_anything_3.api import DepthAnything3

[WARN ] Dependency `gsplat` is required for rendering 3DGS. Install via: pip install git+https://github.com/nerfstudio-project/gsplat.git@0b4dddf04cb687367602c01196913cde6a743d70


In [3]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

cuda


In [4]:
da3 = DepthAnything3.from_pretrained("depth-anything/DA3-LARGE-1.1").to(device=device)
da3.eval()

config.json: 0.00B [00:00, ?B/s]

c:\Users\ASUS\anaconda3\envs\pre_thesis_gpu\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ASUS\.cache\huggingface\hub\models--depth-anything--DA3-LARGE-1.1. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


lib.depth_anything_3.model.da3
lib.depth_anything_3.model.dinov2.dinov2
[INFO ] using MLP layer as FFN
lib.depth_anything_3.model.dualdpt
lib.depth_anything_3.model.cam_dec
lib.depth_anything_3.model.cam_enc


model.safetensors:   0%|          | 0.00/1.64G [00:00<?, ?B/s]

DepthAnything3(
  (model): DepthAnything3Net(
    (backbone): DinoV2(
      (pretrained): DinoVisionTransformer(
        (patch_embed): PatchEmbed(
          (proj): Conv2d(3, 1024, kernel_size=(14, 14), stride=(14, 14))
          (norm): Identity()
        )
        (rope): RotaryPositionEmbedding2D()
        (blocks): ModuleList(
          (0-7): 8 x Block(
            (norm1): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
            (attn): Attention(
              (qkv): Linear(in_features=1024, out_features=3072, bias=True)
              (q_norm): Identity()
              (k_norm): Identity()
              (attn_drop): Dropout(p=0.0, inplace=False)
              (proj): Linear(in_features=1024, out_features=1024, bias=True)
              (proj_drop): Dropout(p=0.0, inplace=False)
            )
            (ls1): LayerScale(1024, init_values=1.0, inplace=False)
            (drop_path1): Identity()
            (norm2): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)

In [5]:
BASE_DATA_DIR = Path('./data/RescueNet')
splits = ['train', 'val', 'test']

In [6]:
for split in splits:
  org_dir = BASE_DATA_DIR / split / f"{split}-org-img"
  depth_dir = BASE_DATA_DIR / split / f"{split}-depth-img"
  vis_dir = BASE_DATA_DIR / split / f"{split}-vis-img"
  print(org_dir)
  print(org_dir.exists())
  
  depth_dir.mkdir(parents=True, exist_ok=True)
  vis_dir.mkdir(parents=True, exist_ok=True)
  
  all_image_paths = list(org_dir.glob('*.jpg')) + list(org_dir.glob('*.png'))
  existing_depth_maps = list(depth_dir.glob("*.npy"))
  
  existing_stems = set(path.stem for path in existing_depth_maps)
  image_paths = [path for path in all_image_paths if path.stem not in existing_stems]
  
  total_imgs = len(all_image_paths)
  completed = len(existing_stems)
  remaining = len(image_paths)
  print(f"Dataset Status: {total_imgs} Total | {completed} Completed | {remaining} Pending")
  
  if remaining == 0:
    print(f"Skipping {split.upper()} - All depth maps already generated.")
    continue
  
  print(f"\n--- Initiating DA3 Geometry Extraction for: {split.upper()} ---")
  
  for image_path in tqdm(image_paths, desc=f"Processing {split} set"):
    raw_image = cv2.imread(str(image_path))
    raw_image_rgb = cv2.cvtColor(raw_image, cv2.COLOR_BGR2RGB)
    prediction = da3.inference(image=[raw_image_rgb])
    
    depth_map = prediction.depth
    vis_map = np.squeeze(depth_map)
    
    save_name = image_path.stem + '.npy'
    save_path = depth_dir / save_name
    np.save(save_path, depth_map)
    
    vis_path = vis_dir / (image_path.stem + '_vis.png')
    depth_normalized = cv2.normalize(vis_map, None, 0, 255, cv2.NORM_MINMAX, dtype=cv2.CV_8U)
    
    depth_colored = cv2.applyColorMap(depth_normalized, cv2.COLORMAP_INFERNO)
    
    cv2.imwrite(str(vis_path), depth_colored)

print("\nOffline Distillation Targets Successfully Generated.")

data\RescueNet\train\train-org-img
True
Dataset Status: 3595 Total | 2 Completed | 3593 Pending

--- Initiating DA3 Geometry Extraction for: TRAIN ---


Processing train set:   0%|          | 0/3593 [00:00<?, ?it/s]

[INFO ] Processed Images Done taking 0.2084653377532959 seconds. Shape:  torch.Size([1, 3, 378, 504])
[INFO ] Model Forward Pass Done. Time: 2.5048394203186035 seconds
[INFO ] Conversion to Prediction Done. Time: 0.002007722854614258 seconds
[INFO ] Processed Images Done taking 0.11787748336791992 seconds. Shape:  torch.Size([1, 3, 378, 504])
[INFO ] Model Forward Pass Done. Time: 0.6950654983520508 seconds
[INFO ] Conversion to Prediction Done. Time: 0.0 seconds
[INFO ] Processed Images Done taking 0.08548426628112793 seconds. Shape:  torch.Size([1, 3, 378, 504])
[INFO ] Model Forward Pass Done. Time: 0.16199541091918945 seconds
[INFO ] Conversion to Prediction Done. Time: 0.0009999275207519531 seconds
[INFO ] Processed Images Done taking 0.08305096626281738 seconds. Shape:  torch.Size([1, 3, 378, 504])
[INFO ] Model Forward Pass Done. Time: 0.15632271766662598 seconds
[INFO ] Conversion to Prediction Done. Time: 0.0010056495666503906 seconds
[INFO ] Processed Images Done taking 0.079

Processing val set:   0%|          | 0/449 [00:00<?, ?it/s]

[INFO ] Processed Images Done taking 0.13570094108581543 seconds. Shape:  torch.Size([1, 3, 378, 504])
[INFO ] Model Forward Pass Done. Time: 0.15439224243164062 seconds
[INFO ] Conversion to Prediction Done. Time: 0.0010063648223876953 seconds
[INFO ] Processed Images Done taking 0.10403037071228027 seconds. Shape:  torch.Size([1, 3, 378, 504])
[INFO ] Model Forward Pass Done. Time: 0.13582801818847656 seconds
[INFO ] Conversion to Prediction Done. Time: 0.0020012855529785156 seconds
[INFO ] Processed Images Done taking 0.11057877540588379 seconds. Shape:  torch.Size([1, 3, 378, 504])
[INFO ] Model Forward Pass Done. Time: 0.13928580284118652 seconds
[INFO ] Conversion to Prediction Done. Time: 0.0010035037994384766 seconds
[INFO ] Processed Images Done taking 0.12241506576538086 seconds. Shape:  torch.Size([1, 3, 378, 504])
[INFO ] Model Forward Pass Done. Time: 0.13806772232055664 seconds
[INFO ] Conversion to Prediction Done. Time: 0.0010085105895996094 seconds
[INFO ] Processed Im

Processing test set:   0%|          | 0/450 [00:00<?, ?it/s]

[INFO ] Processed Images Done taking 0.11678194999694824 seconds. Shape:  torch.Size([1, 3, 378, 504])
[INFO ] Model Forward Pass Done. Time: 0.1404128074645996 seconds
[INFO ] Conversion to Prediction Done. Time: 0.001001119613647461 seconds
[INFO ] Processed Images Done taking 0.15293240547180176 seconds. Shape:  torch.Size([1, 3, 378, 504])
[INFO ] Model Forward Pass Done. Time: 0.1380760669708252 seconds
[INFO ] Conversion to Prediction Done. Time: 0.0010595321655273438 seconds
[INFO ] Processed Images Done taking 0.14492082595825195 seconds. Shape:  torch.Size([1, 3, 378, 504])
[INFO ] Model Forward Pass Done. Time: 0.18808937072753906 seconds
[INFO ] Conversion to Prediction Done. Time: 0.001504659652709961 seconds
[INFO ] Processed Images Done taking 0.14241528511047363 seconds. Shape:  torch.Size([1, 3, 378, 504])
[INFO ] Model Forward Pass Done. Time: 0.2202467918395996 seconds
[INFO ] Conversion to Prediction Done. Time: 0.0009975433349609375 seconds
[INFO ] Processed Images 